# Pika H3 Flex 官方网页工作台
选择 A100 80GB / 高内存运行时。若已有测试正在生成，请先等它结束。
Colab Secrets 添加并授权 `CF_TUNNEL_TOKEN`；可选 `HF_TOKEN`。工作台无需密码。
运行下一格，通过你的独立工作台网页连接后端。默认无需密码；已有环境与模型缓存会复用。
Cloudflare Tunnel 服务地址应为 `http://127.0.0.1:8188`。首次模型下载发生在网页提交任务后。

In [ ]:
# -*- coding: utf-8 -*-
"""A100 官方 Pika 工作台一键启动：复用模型缓存，启动网页后端，无需密码。"""
from __future__ import annotations
import base64
import gzip
import json
import os
from pathlib import Path
import secrets
import shutil
import socket
import subprocess
import sys
import time
import urllib.request

PUBLIC_URL = "https://mcp-mac.id96.vip"
STUDIO_URL = "http://127.0.0.1:3091"  # 电脑上的本地独立网页；部署后可换为项目域名。
APP_VERSION = "1.3.0-2afaef9-dirty"
PORT = 8188  # 复用原来 ComfyUI 的隧道服务端口。
STORAGE_ROOT = ""  # 空白自动检测 scratch；也可填 /mnt/local-scratch。
HTTP_DOWNLOAD = False  # Xet 下载异常时设 True，后续生成进程走普通 HTTP。
BUNDLE_GZ_B64 = "H4sIAAAAAAACCr19aXfbRpboX8GoJ4egQ1KkZHlRwpx2bDn2ixPrWU6m+0l6PBAJSohIggFAW2qNznEWb4m3fu01djp2x0nci5fuzjiOl/i/TAsU9an/wrtLFVAAQYp250xyEoFALbeqbt2t7r21MmQ1XM+o1Uyn5LQanlU3c83loXFt6BfaW2bDdAzPrGhVx65r3oKpHTEdq2rBG6jTqBg1u2FqQQOvaa7ptZqa3agtZ7SGrRktz64bnlXW5rkpy27kZhozQ/ivf/usf+qhf+qBttuuGXNa5/mFzq0z/3z6hX/vavvyj/65r9YvfqXtPrD/n0/PrH/+l/U/f75+7r7/h0/gp1uGtsoLmn/3qv/xnX8c+5hbnGkQnKVSteW1HLNU0qx603Y8zWg0bI96d2ca4t2C4S7UrLng9weu3Qh+2K5oq2l4WEo2NAk/g0LuQsuzauHP1lzTscumG/bhLofPOLMzjd0H90yUDk28rxW11Na56g7TGMvv2LGzYGwdHSnkt5nb89Vt28cqIzu2l0e2V/Mjo/ltldRM4x2odkDWy1cq1bExo7I9v2172SiY27ftKIyMjI3OVbbPbd2Wz1eqO7ZXt4/tTGF37+7d/xZUWkm5nu0Y82bJsW0vNa6lUhkt5RnuIj57I0cM/O2Yrl1r4Tzh26078k18W2nxysG7MfhpuE2zTE0Uto3vxAKuaVbg99YReC43WyW7Wq3ZBr467LRMeAnzUm96papVM2XXVctxvZJVB5Dkq5oRf+OYVdMxGzCn8GZ6Ft4seF6zVLGPNkQHe42aa67ONN56EwZZyGtbtmg7AcEaFbOqATrrhjPvZrQtGa18tFJ8F7A1o5mNI+KpZs/TU3p8pqHBP1gYmqlZrqfXjabuek6GXqbTXKDpWA1PT/07wqalch/YFveQzmjVWstdKOJ4RVnYE7DqRVr3XN0GBLQbVlkXX60q9q5ZroYAiP6pWoBFuRB+BB7+Y9jhP3izYJYX1e5MmAelmaOWt6DZTbOhQze4aCmsXLYrVmO+mGp51eyOVFozXNgGjUpNrcnjtMsAugLLJLXVCxrXq9gtr6iW3z85Qe9Nx1HfTx3ec/C9wxnNM5e8ImNHHCx44zi24xZh+Zs1o2zCm7lW1bV+YxYL6SignrMcAx3/qdoOLCJRJhpKjuFLKBkuKpZHWCpFxLyu1Yz/w9OWO+pYnkl1+5ej9vSEMg7OM8F41LC8pBKAKU65B+iOYbmmdojJ9gTOml5N+b99tvbktn/7r53vv9HMJcsrrjjl1X8+ve7fO9O+9H37yjf+8ytAZlcAM1ZTsR7NpbLZ9LQ3DdecoEfY9+OJQBHQTbtW09MJaBzHppxnOnWrAcxE7zFTyUsZaYOmCIeKyFYY69GQGIKCdIe5ysRS03LMyma9LFo4pkFASVpPXJHo6yoMu1ZLGpyCm7lyzXaDuWGUrKY2rj5s3/uvzrEr7SsPccXitETLMpkZzxWqq24cb5kKIqEsle1G1ZrXJaHjn4B5Favs6cwkZNcGEFykW8jq9BLR7FIpnSPWcAQgzIkSgALzNXsO6C+8mzeBLMrCQFiQGlELOSAUclTlKnYpqg8DHyuUmNUDn2+furBx/Oz6s3s55MOpgEhCnZzlUsMB9Ewmm80ayiFFYtw5HKWrY2nHhPEiedFjhCXrWvOptLJm0DzMJCAvCTDAZXTZaoYmJq0BHQGJJnhN8w2/efrSsRXlzfi+UWuJrZjiEfnPj2/cetK+et+//2Pnr7dAfFl7dNE//6fOJ8/Wnj1fv3hHE1x67dHd9S8+9e9egTVX9yV3l2s1K7h9AmDCAgG6UH/tL79pP76A6ALTAfv+VOfbE+vXL3c+/aJ94y8btx5ufCJ7/OfT00m0zlxCBk9zu8KiwbimE6PNML8FdpeyxHum4cHrWrQ4tYrcvhYpLl8Df09qfjVcfhr6NAsqs7xcjQBAZQG6Jx+rwJSebV+83z7zUSrd1aYi7AQtrwRCDwhVzdRq3x7CBjRYTu5Hw/qA3FQ97NNbbpq67DiQp2aJcnLXHiIbPo5prxe17qL4tjDWFx5ZWAswThv772O/K4xpgFZI+S89SJgGIc4pUzBSYLFOinej4yP4Z+v4KP4ZG99KH8cL/HKMi9DLES6yc7ywjf+OFDaZQ+49cZ0ic0YiZuJ85dX54mLa69oISoLbRvv2jYW1zplP/OvfY/93v16/dU/2jyLEormME6JHZNouGVQlAzGooYEQ5Dnbrm1GMaqpFaizGi6g/+gT/8FF/9hTZV50bCmYGFWOnoU9FfmoSNSz6bT2b8Vg60xHd9Zs34na+Pay/+An/9F37csn1548ZOjWHp3TaI8BWfMfX0ycOFXuj8n8UXk/NonK/IHuVtF0nEBiSOrMhowhmRTvhW/v2t5eu9WoROZ3XFtR2glkIKBGKP6H9CFQPmaDyY8xDKySIYWBmIUeo1fFYkDjaPZpcbBOuu98rz05rnE1iQnnL6z9dF0LIdL8U1c6t+6okw4fcdKxdaXxRasB3Mw1mxlSZWF4UABZuGchqdBT46lujgjFcTRYN6QKwbIdsSqmjQ9Gq2LZqVUaeLBA2MumKxNBef/8x51jnzBqta/cbF86hbwLwFQWBtT5hlgYKdfMw6ZsOiYpQoFoYDWqdlRxoe8lkLCaLU+fTjUAesvIunULR5DNftgyneUstFWEwZpLmYZRNzN1s247yznP9oya/NFyQSyoONYR0ynBfy4SZWoB5r5ueMWyeyTTsBdA+DAdeGg1LA/wRlF0oirkW5PvadyjRl1q1Jn2jvVmRsOu+In7G5+ZaaS0V2lwCeyadhUMGj+DNOlYTRDIXJARPNRLQDqbzs/ybz2VUegI0jOQGrD29AjQqde17WP5fL4LMyPqRcp/9jv/9FkN4UeiefX+xh8+bd/4CjiMtiMPWnj71tcg4XTu/wDEtHPvNht1QHzt0fEodPyGtjW/c9sm/WKH/g9/88/eBImp8/B45/lJbetb1pv/OPYxdOYfP+V/dKN99w/abrteXX5vP7Bg//jDtSeX2+furF844f/4cOPkZ+2LPyE+m7ANF5En0ijad275v/8cYPZPnN04ebbzw29BVQJJieUlGGPn5J/8z+60vzzWeX59/c7n0KEcC6CGlKFJB5SzPA6YUUhH1FDaG6lhxMphqIZrlUorsmp0xaSUahwxrJoxVzNpeT0dak6n3jHru+T7lFxZXOS0tkUr5Ee2RnUI//x9mLH1vz9Zf/KVf+K4f/cqbq+w5WHtrTdJf4C/SdIgLFhY+HWtgEYWrLIJmlCv3B/O4cO/Y8233mTcgOXi5RBrJJbgeuf5NWR6F+7DhPvX7qw9/9K/d2rt2bWNk+dxgWARv/w9qwztG8fWnp3rfPtR+6+0HEwTXLMG3K0kTF0xOV2ReSK2MJX3lRdAC2tI7Se5eDoHTBS4EmxTR1cUoy4yyo0hLaxYjp5MCaOzpnaE2wvtmzfuiDm79Wf/xDX/3u83rh1vn/m48+xZ54f7/k+fppINQKArGFHlqDdRBA21Um94TM9I/6In/o7Ph3cdemvicGbv1OFfT05EqZqqp8CkWKijIBednsVFCaCBxTkCfNPTLc+su/G5wJ2CH3CnUIEEZRn0XNAy5dJgKWSz+A6WJMEGhEBgKdZNqy4KZjiWVLKNpeqGrM6rN6suWT7NI8Ezzh0ZO5ddfuF+2DLcBXheJRklVQWESMlGqi7M+lGze9kHM3e0XEADZGRkYs5VLHexRO90HnIf8wSMhUoy+9LeKGqjed6yBKbt5owyIoFoKYNv/qN08O10H3Ciq5szmk2zUdF17qfqmMi/GK7+NpmDU4To/SwwBprP429D1OHVLC9YtQpIQrAM07Nqn1wQcR+xumbCYiE6RTBE7s5gPPGN2WrAnqhUaRMsHO7CrqSN277xp/bXx9r/9bl/6gGQPtioyF7eKMLUI0+88Rf/xoP161eZWa09OadVqlrnp9/5Jx8zm9TkiQbvaaB0vN01lSIAgQSasPb0C8GRLv4IJHbt8W+54YjFQJKxurGkhwPNoGRerBn1uYoBM2054/R/ZB3TBSHj9sY8bjTkCiBqyHc5skK5aHQGJgdk0zMb3jBJL6n0JpwCJmTj2On253/kWdKiZztyVEdtZxFlcx7YsJaatBaN0r7R0t6auVQ6WK1aZcuoCbtR2QACh+aLfKg0UQNQb6GaSieRZCRBVbuG0ptWAjaIyIMbGXbHUaO2GKmfQL1QiKN9j/WSTH0sfBPt4m7S0BrWSqRGiKBYBSF1l+sgFiy+JCURc/FqkduDhSJhA5ik9RvzZ9usjvlhC22rAuVGxojgZLSRMUF6sgKQmIGTF5lsVbSyq9p/aut/fLxx5W/aSkhdIjIKlNi3V1t/+juoC3ocD28AIUZp7vUA3v64WU0RLN+zAIMSCAm/IPGu37vCssf642+1lWD0ETh5HCoC5+qLiHbmEuiLJTtyfCNUHCwVCjNleKsjjsilP4J6E8wxIKUJmowDvH1eFCHOJsX/iLpKlZSBduMLnXTO2/Z8zcyV6SxWnFmiiIPENFpcQqHLzwEQpAR3waHgVK8zBRWTaCKoDzkTZsPF49wjMGYd0MQrNZcz2hE6eaL/1exAyGsuA2D4CffqnAV6E/xtLnsLqlG5uZyjNXAje6ppNUskGEUVSGIHSOuay2iyzNZJGLCaLCNJVXBWPZszmnT8zPKTOOWK64ESFNlpjgdetismmg3ycabE09JcFsIezQhU7gGrmKUQ4KDCvwi2VQ37VkH+tyjIUAxwFNDCbFkVnWwfeWmwE5zl6IJVBm4BnWZRhhtEPF5/+th/8FttkpZTG80VRnip2XQoxOR7pztfH9d4ybFIlor4F86xcpfqspunOre+2Xh2IV4FvQ4+fQj1/FPX/SePAY/Wnt/zb5/tXPui/flX7IrQ66SQRQg5NJhxtt3T3H/4YWTW8Qwa8bdPZeFfQbWXZRsZidYBwH2apSaDnSNQgupkiI3jY3qT6krNvmgUqxriLe/luRaIbqWyXa+DWKKXa1YGyjTtjNDNQBwl9Av2M1kQFaucalKcpQMagB9bwI1uLhn1JjBg3vVUkp51YYXLeUteKuFcQbHT8dEVHXtQaSldluvI3GhzAdC0sbJUOxNrjL7UYU/UsggWThTKKtkDxpw7TE/7RrMotfAUOibKrKRyBU4e9IHBzwrbKf8SNYJzi0ziYQaVCk4OeIUTzh6omLD+Z7qOA+grmdOjDQhDfFRBxAJi3Wa7lG3Veh7RtesVqUlAW1AqK0tFTcm4x4x5MijrcRNyNksvssKKnIGVVqzKVAB/y++9zcwx6oOwAeVD2Kax94xadrbb6ppkM44NVTZHK4jKBfx/NmbgrFfkNmkC7hhAjAWbr4M8rSuGDNBrcs2a4aHxEYlrCiTE1lJqc1E77nR1AOuRHcw/e8k//kn75lM22aVUNxYS7JIdWfocJUcttBE5PtlQE06rFKdhOe150rYZvegRVHJBhbIk/tGZTFQsl+I6CSV9xC5kDRFxqmw3l/Xwozx03be3tO/gOxNFRHRVE8ho+OW9N0u7d+3e1/WZ/rTmRLFfTRzuXWwJWWDQ2p79U7vePDCBVYqpQoRiRQ+fZgW5ysMcTO6f5Pah+qFIH+FMQReH35mMf8cJhS+Tvz687+C777375nt7904cmtiDXWe03e/t2VV6f//UfgRoz8T7+3dPTBWpQ/oCROs9+HDg4K49+999q5g6sOv//DpFbR08tHtficrsOnDg4O4S4kYxxbYzNCaWXHMe8dodxwWR+ObZi6S5Cuk3BTNy+ODbE++qB4NYRFlsWKfpsNwsoiqWCPC33pTYC4Ou4oOeeuXXr9RfqWRf2ffKO69MpdLaq0Aq0MaO00JF8X+lBkiI2itaIU//CAjQb6uoybkj/ISHagq4ZHaFulvNwdtUTNP54Zu1ZzfWr9/zn11CfQfrr87MNGLuOAkCRd1yXauBfU7Ttojuj3mLZIQm6sx0xlatN815fJovk72qbizC5AZHO6rwRbsjJNiio03EuIHlNGFy/uEb//gPMD6cXYSHveZEX+n/caGp93yRZJIFMRpwEm0JoYSVrZj9pCshWCHSAlk2l8xyyyNruSDUQkoqodF/enyExA19NKMVRoSjTFQiDsW6VFSRE/1sZnkn+zhrp6qoHKivaKs6/ROwADS4B8YXPmFURWu0YKmeucohDslb4R5w7ZZTNlMRYLFIkpZF75kgdy2fWBurYUmhAkunB1l5UTW7W61IHiZ1m3HIqNDhve1Y81ZDHuO748PDUBWINFD++nCyoJbDxvssv2lUEtSwPiAdyQJvdwmqfRO79mDbmyhf0kssrt9T37ArpVuxegBgOd5y37PQXgCijajlsvTUtGFla4bVSDrLFEBQTxK0AchD+/GF9ZsfMSls3zjNdr/2xR/9z8RRx9rzL9cvXQPVq336czwJuv3p+oUTofWT3JlSA6NA1RRbHYXFmmc6xbmaPTfesBtCQqyYTY8KFCIIIuf0X0E/mnG75cmOPINA2bzpIyyZyB2GP7PN5VGgCYqdYxC7SCDKTKcmdx3eR+xRqnzSRoLsD+g8Ggjx6P9VknzI0E5VlEOTKtBs0yGeBu0IF/qcu2CMjG3T9cC5nflocxkb7ol/ik3l/diZUo5c90wd6wfKnbCykcjAqhmfn84te3RmmlswlyrWvOkGjpl1w1k0HWUaFWk6i3WXVW9DegGFyQctZIrURqIPYrchTTahnLqJ+spJb5oO3OQ0svapvOiymemht0RGe9tcpqd0TwMa+gsjGJFTzReyalHxl7NoEVvt25Nn2zV3GPS+2nKpYlXx0MxxS1TPBBRkVp0NQEBfc7YPvDgwcjUS7Wt0GEbdxr/E2Fh0KnuYQlRpI1uxXGT/WRS7Be8XA6LPrea8Y1RMpTZFzdDE4K+jC6ZZG0Tc+ZfAcFJseEncWckT3w+MTZf1xZvsPbKGjTSbOZQpxvFS89WN6dGqgMNzaOmeGZJBPMAOF16ThsPD+Gs8laG3uZL0CyqVMinUe4Iv4n2u3KoY6dc02KqmbIveIXUJfCz0tGxfKQDEuARCqFUGnQSEdj2fhlJLRS5ho8+IPjqSGR1JZ7hUMYXVUpkKnnqLYnPkd1PYFrSvL/1yCSSLVl1P5/BsVcdGlU7d5UZ5Acil9Rv0bhjqsgWW2S41Z/aewHLNSrLHo3C1MIomrpTSKhkDYXkXzFpzIHwRtJWCMJi4EuGttOpNV1+JUNlxlcTiysNuLwW2t/GAI6+mE+JkIkYa0poFg0VL4ExjKKMNwVDmHSB3GEPXAJg4hI7D0g7OwXofMSl+zhYnlWj4N8k5CA9NgZRpZsXyUM07alrzC54LcgSebLXwGJK8I1yNxeycjHUTKFltNcq0+5Ij2TjsbKYxeWhi7/5fES5PT+5/e1dp8tDBtw5NTE3NathYcNRStzy9CYwdaGodxmPMw8OWLXQc44ZGWUQf0eKrmjLpZASi6sVoI8XuxjJSfjHcsmUVpQt5LIRBnIV5aCaiocJ6AdpFGpdw/TKYC8AJA8CRFcQKYkv4oWlW9C0iMmzL4lGK4FLtCd1zEOEqhAay6a6Goud53JsyEDRQwzKXbEYKRxmUst9eZkQixBKlhD6DY6+QolYzGzp/nZ4Zcq35uuGWjswMzWKMQ0GNPWiRqJdXQi82sQgGUwjtitHOwBaZGfKfHvO/+5z1042TJ9s3f8D3oHc1ayY0WMwLL8gi/T8d9x8KCpZcz2zqdRs3R0azGmgXjJ8byH9Ayq/ZZRgxDST67RfaIZP2iGgZXeeNKigIZA8lmRGIhuUuoOcECgN10zFry9pu+ArMlYWHaJNRkhk1KdNMvlpUZ5f9tmBdE6dTRvcg/kRqdE9udWaIZxQ9m6Gf1eEVmsTV6AzTt8gsZxJO9wMcdc2y3ai4RQc9qHUBaEYDLtOvFig5yTWB9PMkbNF06jxLP9PoLBlZbQ6agxkJUBpktHkLlt0pVW0HsLZSWrDtRT2KEun+onl02/bZLYpTSGLolojpQ9NC6PfXa8PXQXcOtAa5criq4bY49SVuizOXhWcjOayu3/ncf3x+Zkiq+or4EXmDrLS0MJoL2InhBqxFqGwUxiyKlW3HlFVxhcgJHCDJUJgY+r6YSJ0NJkhHDNGGbFE6O5ZQhPFcUiWRMPf4jqPjJx4pukh99Q2epD76vPPsmRgvOX+uP/l07cnDYLwqbLl9o7vlzwkS0x2hHwb9D1YcQVBLMlDrTy+v3/yoff7C+u3Hnftn1x6dY4f09e9vtr+8EEDE85Pj40vPMRounr2Qdskw9CqAfeC3cL0/uwmDX3v0hEcOHW4c+3jj+Fn/2p2gt3AVcrSvzFL4plQzKhXuuIuZDFAv00PEk4YHw4QZ3GPSxLkg1KI1rlQKhtnjOw6sYrLcpA6z893XMLvoy/voHD9vfPX39YtPgpFieyzA1ZtbI70Eb7E1KZOJti+cap+6oL0zuVXzf3zIaBVrN4r2IIYKrMf9iJtECmIlfMFVIq+CQyDeshUQsrnvzncnNv7wW3RNRs/mc+2btzb+dAZDa08xsuC/oDyWSFDHiUORq0RtlkozQ+NSdOUuUHbktYuIjGh+DGXF3Qf2ax+2zJZJSissrJllK5VWt+csoJSuh6EYOWm+5dwMTkWrWXOO4SxTQobcz5AjYaFulJPFzDrwLtQ13BdLoeCYYa4EOvBxe2ZXsOaBFvdLtuAtoLKOOBKVfQkINPaGhM9etExX+ZLjJZAFMPp53+HDk4dALzZdbx/Re9g4h2UP+HGKqohGWk4NDWFk1g1IM/4ofQhUEL7SD0QNNZPDzNCgqRxw3XZNTpbenzg0tf/gu1S3kBvN5bMjRtUwqzuzZITFYod3Tb09haGbM0PoxAAY1xW8Ceqs/BIN3wRKFavCkZoK/5sZqtailUUw58wQe1B0d7gKsE9NTuw+jGCBtIlxhbyXMFCQnwrjBX7A4EN+Gh0f4Yet46P8MDa+Vb4Zk2XEmxFZBmMOQYydaUz86vDEuzhZYjLIBQCBg+dcU1KS3AfN8MmUj0fNuebM0GqXmDMzRMFPshWgTaICSAJh1TpWRdaHm1KWPWrIEvWmADVXrcF2Ei+3wsStwkzt30MxWngc0USDozMz9H+n89mdRrY6uzI6svrvRGTe2fWr0nuTeOgKpUfIvxsjP7ZsGWECVK4B39WmiCyMh6qCpNa6a9aqGQ2dikmJRT0edGHzqNDC1EBrKJnjgvRIpelJVJFerVgGD5EpcII8ZESBJB9bOrucGfoAGBdPgDje5x+tJtlOYaQxyUsPoMETV3ajFYdJHFbuypwSCWf9wWjQVgrSfYVP/vF0s3EEgCGteOrwe3v2HyxN7pqa+o+Dh/YwPAFTCZpwgfAgS8VIR3lW7ebo2FnYo0dH4nVAD0EjbECmcocO2KFjRFDsqLFoRopNHCHnjzgEno2C5gAlcZJxC6zG3hsgCB/BrjB1Q3yKOAFIwpeqYdWAb8Rb/IXWvnJz/bOH7WMftU9/17l1Blj+P59+sfbkif/ZLYyO/v6Jf0PEA22cPPMPkHhOnsVAnwvnNq5+5Z+60n50fP2Pj/GQ8oe/cT4APH9EDr/25CxUFfqdO4zZfqIg8QREZoIptQhxKPK82mgsymgVA+T2RiJicH12ZtcDjYTVdpD0edPAdKpoyVw1GLx/7/fr5x7AyDvPr8E40XP4wrP161cx6p9OtToPPsFwrI9/bF+9v/bo8drjExpDhueta09QQoJKGDt16oF/+9P2xR+V9EYKQIyABFPEdoE5LpbFGQ8ZEaUzg57G05TtQCl2bNsaODMoqouo+iqSo5kh+IvcPtcwj+pxlM+IssExTSZ2FBQ7iwnBBmHUAmVNAT7Dzhr9z1W4uwwKAdK7Q8S9EayoQHZVkCkMfrZRJGiTOMHcCEYxKlNNghr1jHQcHblEOy66OEro0v3PenZ5nmPNtTwz8cRHgCBOq8IpZuKJpvcFMcFkbwt7jaQ8Sg5rFjUAh9LSa3f/nly1VavV0ejB3zcN7QXdinSo9s2TnfsnZGgvHvDC3m9ffrD26LO1pzcjxJWOIGUKpgi1V7lCDhOPMBCMrlvgZUIr001iOU3KA0Tv0OM657ZAsF7CY3OUBGCvk7ygzAbaxqg8ebwUBh1nLGCv7yA1jsdwp/PRrgPEFcdTCwJaGU0m48tC6WZaVpkdFM71vz5hvZOpM4J95kdQQCNASsddktZjG5jSkBBqofd/xIdeuusGYQEzQ/wugZN24530clURr4D5HWhF6KP0M0jj6x3ReOUeo4a1wFir499ohf8+9juso/l3r6z/5VtMiyH1/ghklNcgMgh8w0NgobrvMLBwOAhqTCwbCecDrBMxQLQPPD0fpMaILY8bBTB0ABYi8o58M1IjyA4SqSbfYiU1oZLIyxEpy++EvE4yfN9ZcNBsKicB4ZXxlRI4bAfzpIDgS/OEmTMkPD0zowTDoHwo8pPSLUMZ9ixGIjoXishASCPD0k4D3cLEPKdOdH66s37u5D9AxrnycOPSc9hC6xef+D8ejwmIxHeUicM38cmiUigFcyo63BoZLVuIE1XRmJQwHSALcybQAn1ky5aIiClTj2CFnmlSqDX4C5XHRtHQPwAqfgdE7QLl+3gMjXRO/l3bmc9vL+zcOTK2dfvW/M6dhTC9zD+fXgcRBwOfvjjfvvE4Mmbh3B2dGcU3PD5BsnxybrFfaGioO312/c8gRR3rnPweRC5KvQM0rf3peV4fWCjEL0Cc9e/+H/CfjRvH/LOPOs+eAZz0oX39eyjWvvX12rOzG0+udu7dxjOKC/cjMmYUeNp4xQBziU4HOPmGNta9IqLuCyWCmRnaTS7ZCKx//mOYWmafkUki1/eMhg7u0VlVnOQDsptRC4Q+8slkmVPMUDMypQxW4RQyRMOmkarNDrSN2mdOt08fw80k6BkyxCClTHREgKzM1nFnKOOLbwur2hXWFpHkVSFICDOBBSBCRatdZFQ68OPMTM9uQuGi2V+YbRQVU0gs6wuWQlbGP97QCiObz6B/+wuRIYWmD3CY2FhhBHbAn5iho9YUJFIgjtH59qONa5/6j75NtGUHx1FuzDSSz6gmjnzEiJFXtL1yzTQaRJimZ6Orl5yFpuf0KYnWuj+KJcE0NITAIS+JfZLEncc03iMrX2xalYlhgShhh8kMOpw2Z1p2OBun0TGMCwG0GHKslnQc6E7jl9mEU0ExwzJAZkV2PS4y+nDT4wIsfJ5dTceCW6iDYHVnAd924vQFH8RC44fRyAex5sEHwleidBQ439Xwq91tDqActIEa3/4CYPKv/wQC/qjGlnV63Pjq7/BIeP1XjVEdvZVP/t1/8Fug9/7Tm1Bp/fTJ9qnL8BO+ct0kCZZdEkh2LEZFyAyJZkWW1kL5qUiCi6TpRfmQSY6DZvGiGEgdwGGL+L+MpvC2ovib0RTSXGT6ltxsSKGL+JhRcj8VBW5EDRStOTyoSJbKSdxQLHYyEFPY8xKdG3s478LGISdykAZEVoNLZza+/MPGHz7tfHeKHcrRWMN+4yKRi5LjsX3jLGgcMW3IMeouxwRVc4F+QWNQ7UtrT875P/wNpLGNSzf829+jw/rj28D123/DHIr+hTOMCJq7gJkSzoj+Lj+ARV40MaXP5Qf+k8eiMggOwXerEjUpQf2SFZfg+GUEcOHVFS0oXm4iGovm+mjYokSXOBgAF7U5LphLemFbgjgo4IlLhBL217tEwHBUhVg24cCI2WWcAQbIASqBvZFmQ46hiw8ENZCkyB/TdC7utVykKKwm0JlXhUUU9KsKzv/KOIs1dn5YHYzed55fA+wQy3/3a0BSxoAgIZF//nL74Sn/xFnGrC4+gJuoVdc/eGk4iT9+gFXCaWJXKNRm3wBtdrCRsL0U98KTWyLV1t3T/k/HgRbyGFit7xoAdCjD9CygURIFxYIXxV+gW46JOd+Lijkr0zMHBE9FUZ0DOsGFNwwV+tLgW97mRf6TTmCgOB/TAip0UoffCZbxHOZf7SbyVDikhmwrVm1gyVbb++dRhjp9ihJ9IfXgLHRAVaS1+XUB0RvZI6+LGXoDz30Uo2wMlKjRKjjXwAOLmaEVgGM6ZVVSs6vZIytoPoQX7GovY6mJHhTSq3y6pI6qZs/3GVJi93zA8uJ9Q1fRvmXAdmLXTV4u2Bm8vBEJKTAJ/TyQeUueOumCoysuoM3p0OQ0qzhzzgyRN2dUJ6XVkbCpWIMjjMTvyuDvgGkGeYyhOzp2lzKqHsucjMSTvXZnEtxE0XKJpIKjyWmnTEtz0yx/CIPJmaokRynBt65dirWlkyvWVSLM8ZMSYs4b1xGCkoAoalNCuNQ3s8n9qRYlanFatTLNiqZDWxJ+l7/EgNleElbn37IqL5P8Hg07l1k1pmOmhaT4biFaY5NK8HmXPhqLQE/QriNB6IQFCSq2GoiOa96tHTWTAtGTg9EDRFS1Dq6vKrrp2XSidsYrGSq6/aLfGQ+qOIiICkSG93E6J0qERlFMYjWjYEVD3wNyQ4wzpDYxUaivPMJ8LuQpXH22a76ZYoW8nBR3ycYSlgFpkszBLbmeYPBRxodpKokRk2cme5BG2Gn8sKrWC5pAmHgxcIT8IeFhcYcTLnIOS/RbuvqTf/dql5Cg3v4gD4J7pv/nQ4qB7wHgM9bgHgCs2CNdHKJAFSMPMUNAU++Xmq5/9qseVweM9Mmf9y9cH9BjkBkeS7/rJHqdXIeTED24Ds+ou/qmo+oo+0oWlH6JaGOV6yYwr0q49ZJh7/IL6Z53DhanyHRC3qbtWkuJqAvF8JKF5rzOyEOaEPl35ab2v/X2/gMHKCMjdiwCpIOPhycOvZO0fahwrysWkm50iKXL3PyuCoEXk4wUB2x7sdVMSpDGkXgKKaOLHeLn9KoPR1ym7SnsSj1eODwIlS7EnXLLQZmkxG/07rOE0MeBgvDlfhiNmBLYHaEL4qML6GkY2A0k7JigzvSS+6Ix0M4rpHt9RmOGE1+cvvSdJhl4k1A5bXSQ1/UP+qlXRO57E/t0JDHiB+NUVqhByLB6p+hjMHqQBMyDaDVaZiLtxgNgrh05Ak720GHmYHWbHnsxAkULFSEEEXUuZFUnT/i3TwoLDblgB97Z7S8/2Th5NpFB9KMznB1DrN18kjDdm2qzmM0SttA3uqtumrYlnppRZqPv5+G1e2/p8HvvvgvCMSUR6fY8U3oGZtfUSSokR4pE6GC14tlUCOvw5FRVXwZG+H4yi8r1+3AnNiXsbwB1c1pNQAlpUBBOWqHUku7Dv3tdDQVr9xL3QPXnpckMP+HmqEC3C+6OQgGXLo+idyJ88CV6o91TaphHpdNSUU9icuneV1Ql3MYFCg+D1aWa9ryRK+lqq2A9+t9v9SLXVQ1yZZXqKENJz8Pcr0mxhZvlFd5cgBN8mRTnaHA99j+NB2rJHc+mN294oJ036E7sJ6/3+kcYTsT4mEoEv16wFSLr0SbgrYhhTGQefcXYHiT+xXhdgmw9UEaDwbNDh6znKIXFEFYKf6nepWmuunxcE1YacAujhmZB14VNR+1upgOIpjcPLuqjhtSoaxE5JKOCXwYGDsTR/vdRszGaff8AHrN1vjvduX+3c+cYw+dfuyO5/cAAUeh808bzBHfBcCr/CmjCjKUFMVH+qRPrp09qemGrNkWNpweA7JX/VEBAcyFITN6w2/XSHba8lwKWpSQKzOOMSGuP7uL1Wve/7Xz3kczjdXoASClQ6mVAiEZQ9cVaqrRJwy9M+34uutdFrfgx+Y7DiC2i58V7EQEl8VK9/2l562XsRb3S59foFsgwcUmivbrnOW40lX7FtDe3eA8yGBkTFxz40IGaDIUbTNoicAhqykhIP0GY4tcUSVGk51ji8v6QdvHTFXiB8fzoMaHG8gWcEV9HwV/ttRKu+YJrjhET6oJXo6Rk49gx/+RjoCN8S+dpPtJrf/XN+o3PmagEYi0HMnjQnq5ohOnp7Db0g51NvsozSLmNsiW8G3/J3RCgbqCP9tkWyU1ssiPIayXohQw/3bMX2S4JNRCLYJRxG2vvSzgHGjwPWNpyGakUhboPOdpEhlL0/ExX6E2mh3DSZSBSrHeIHL3M5jHLOJ2wiVIRIxO86zJqkct6P7U+YWfwJQqhjSbpBFA484tTQABlNXtkC58/xtdQAEGtTmcLs2QkpC545ZPPF8UIOVCA2+1yzZBXLSR7wbDlVB5oC22ZXFv++9i3Kq4TKlFbpOsBX5wTOl01PjOwcuaijncl5DNJNzNoWW1kK+YcTTbjVnNs4cuRKIEni30U4LQaEyiCWfXkGFc5do6HLYXOIHzqyAGFw4VcPjwd/iUgbNPE4NMACY1ms8t8qB5Pi4BnKBY/3hYEWWBwte6J7AxqS7/Q9ti0amhqErG8MkG2i96qFYtz5sg4PzfXFd1BbaIDJnNToBmUCaUwm46qs2Pd+qvbatK9Tyq4CqDqiFBTFUOx5z7ISGeJkXw+IwDvisUB6IMEcpxSh2omZMoJwpa6QhMblRKwvqbdcCWdTSzDd9WhqzHf15I9vNwUfBHzeFllOrIdRlBeA3EfA5i9YvcZep8GD5iNeW9BntTivOPw0ulNamOm4iy24dg1hqdhZzFVtLlZx7/KqmPJHmxyOLtow21Y1Wp8+/M6JFmvo01PmR40jmU5pQo+xYEJK7jd9nykLcIAQ7OgIorR8hYw62wZLc5dWye4lRFZRbMZxK8m0yk0jSkk33AoeT7tktyUhTkheBh633QpUI9sLEyzxaCEj1s4DeQ5nkyhAmijIX/QLkpnmJdBvAJmynb6NCvb0W+oI+EYiMIrmRkVAUcOjsFKOpLpFSqH69B9MoPUl6/bSxx7HLVnhvLRSUDlC9t4HagL3k+Az2+IHNKDOcz/9eP23a/bl092np/0b39HPq6P1/8YDadIuN+N+B+hGTEHkpP7uiFiG8IDe1DA5N2vj7X/NXXwXY2vtE50u6VrY8K5rtiltyYOJ5+DyWRKUVEGakwenHqxKvJtvE43ggtpoZV4yBcxUNEU42g56Qkbq5KFaRatkOqDxkGMGFULKZKuZrRo4K3oTVrABmpMmuuDYENKNrA1v7W7XdVx9yXhHcuPJszOmw46nU5aTTlFsCsaJuVVOmQCl9gkQ+kA9wH1Gjv7D/snjm98csd/+rH/6JGqLgm3Yy4kdScaRT6CJrzscRxpOZgXTebq0EWqAG8hFrmZ0ejOWC6a4zcy1YeOr+izWsl2vdixFitMiN2q+IheFVg2UPCjpGfKLGf3Yi7j7BQwERQsWe1ybNfNuvRqvDcpTkKkR2fb577xz1/p/HBn/c/XpN8rodNo7JoiEr2xu2GjaQ2D7GM1RPATgtyVdO1oP/2Iby0SfnR4Xm1UKmSqjh+EhqoZMpIe6pnheSZKfuig59H5iBccjmA1mbCAZtFqUlAP08KjWQ+I9Lak41cpJoq2yT23kO+hHG4yzf6DLzv3L7X/cqt96QFR9C9ElutHx/xTJzb+31fibqL7l3juR3Ym+eSEOSuoH2JdaRlhKz4mBHP1ibUVlaJO6Ekx67KkEijfJYsE39L9nH/U9Zi2mqhFB8v3qjYNSzL7UlPcufd848o9//4Jzp7AV7kLTO52PShD96TSvKZN0Q82JCRsuV9l93K6ObOSnXRszw52HSWPh3rCShG3eCTBay8isCierQbifzUq7xRXgmmSAlN69TXKqVIcfk3bB50ebNSWAW6jbiIVKE55DrDv17R3jKXsrnmzuC2/dUc+v8KDXI0Lu7FtDIOtoRATWzKURiWeRSXT+Ebm27ZXZiiFlVUWhjSZ3rVaM5eynJqKMTMIjhjXlGRGlP6lQQnrxPxw3JnSMX7AFz1sily6JK/hw9IUc9eFpenVLpclrJmAsnSZtrBVydw4YXP8AtUyoIOYR9aoLIefIyE1QdhUV+3+9lG2CYWNShsRu8q6RRHInN4E73Ac3TigKrm6QAZeouG6WbGM4WQP1XAYUaxIilvYZMNSyAVH5aubN2HPJnIfGx2Ae7GfQTdfbO+90D7Lb7K1hPlJLn4ClAMxN5G7RxjQgtIR56b+zkvofIv7TrgfTo+DIjI72HrxGMYJiNVBRzvIeoRELoxaExwNRjMSWf66yCjvmEp0lKP0O6yr6bHSwzobhP+T0kQPsXEtCjy10e00TNEwRlnkSKBCuXkQFZuunk5yHkajogg8jSxMzygeKed321wlLRKdh9b03pPaf2KF/7ToqH9PZBPth6m98IMqjoeDlwcT2GV/fAlSuQyCMlxYj4vVPRGDKVgUK/ogQr+RKwbzYNuhZEAV0VqYhBfUYOIJCR88bTK7xDaCLuMndEGA9nCQfE5eBsbGkvg0ieRcpBIFKbeCPFztv91q3zjNGiWmqHp2giOgOs9RtVz/+qP161fXHn0Gn9pnTnOglH/+Bzx+uXGsffnB+hefdqVKSDT641C6Y6KE7V/OJpno0fpPQ4tjbc2cN8rL6nIkBllFWxNNxZwP5U3yvU4SdNEXaQH4pLBzKpDg+CfYI7c9YEgixSz7D75o3/iTkkC0L2LIS9tfEAt66tOo/32dZEuIJ5QSdmxuWBkfu8LxeyE1063WmD0B5eLZdEStM5c8mTCQ7+aLpjcKC4qw+waGdOmL7MiJOaw8s4Hc2o0mQaL7C4SjL3ZhNZSi6S5PTbFY2Mfmhq8gAU/n/je8O9o3n/pPz0eN0T+D1VCkaRHWw6IWJnkcIEvE2UtBOghMc/Lg/Nrjc5gZ8h3rTdzYjy5iVohPniVYEulK7KQL3yMbLU1XZ0fgFXdpE7ivamNBDsrRQWL/KXac6cuzaxsnzwd3bLNoKDJE//gQ8wJeONG+8h1PfQRyEW80SDx0sOe7qUfIjIB6hE2+ihgUjVPH+twQB/Ux7upBFrH4hisHZjE8qBXe9nphJN/X9C5OER1PniIelaeIyT6ejonJhIU3fOKl7uy6H5TroaHPiQSVcTty3WroIrloJmwkne7pNENJbjbxKErcZ5gYZu3R3fblu0EGNbKK9OgKPXzEgQr2lu5lOpDTk+U8YkmFabrFgamgsZumgac6LVRcF+V1hgkZR5MEJ07dEaAaUoN4dg+moeMaX+2ONzT8hn7j31UpI4c0OhAaMsJGWWaKU6L7X9iXQHKI7lieFzgF78lIaHf2M0oLCplw1h27WgLV6oo8LMWrJzJUN1vAYSv3UKB3UMkxGuQbl0B0D+GnuJoWVoqNr5c4yT5I+sxMZUs6y3+QfoftJMsB3JwwqhmNZT2qT7yEyqyyIU2MDSc5QubIcgDQba7AEFxJ6ZM8pnPCNYFmHrnaXHqwACpZH+sYicERSK2RpshFFa3T6shwL/zSnX6Bmn6DmqCLv+knE71eiaCip+Hdk7XJCbaYZ7pggzBB2zK8Qnuwd2BMj1Pg6DLHbJIS10fy237243yVFOC8Banac/MtDEWnPCGkHIGkxi6y6vm/XfZML+t6wAvqL3fwD0WytFivFjY5+t9VxuMgnnVxbE+zvmm//6MuA3K1aLk29xzojUsrNCur2RUov5qEV3i5pqDag/ezx3IxKkXG4FdThofXTuLtbq8Ro0CeIoxf2RWix/gGek4N6NDQ288pMXZFBJSQxxONuEuzq3p0m2WAJjGBiMQXLJWwy6XUElzKEpdYsF6694lIP0FlDlpb7OV+p/pz9BA/aGAxqSPxXhh5BYMzL+4JEPKFS/EM8nVulzPfwmWcpC9yRbhczqhUSoYoQEkNUMBFBIDejFbNK3blOv+Pg4fellbfhjdM9yRl3bKDjINSS5T2jZYwtUTpoLiJIlRZenWK48A2Se6gq9Zk9zsKO3ZsUhnvMFUhnhkqjGzP5eHfQrAtenYsks0PSRteEc0vQABKHjrEyOroo0VSCLXC57b4LrjGB7UCoaDie6H7iHtt9Mi1doEGEd7OSkoEbG86HR1WzkKaji3PiMRdE8Wk6yR07hQnIkOw5nBC8WJ44Z+nNoFqDDTDTnm6ksyfK/KEpCPXkIubRbgKZS40STS2G3QPxvjw8EoAwOr4SgAB3SPVdTN5VHcRQNEfvKYJzc7BLSqBp8OcbTiVIAhRTSIT+Ad0i9vCVi4HnWQtpwO8oECQnSD2bvNEBUqFMAw81ko6edBOKYg3eMFLYFb/P2IBEd2GpwAA"


def health():
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/health", timeout=3) as response:
            return json.load(response)
    except Exception:
        return {}


def live_owned(pidfile, expected):
    try:
        pid = int(pidfile.read_text().strip())
        cmdline = Path(f"/proc/{pid}/cmdline").read_bytes().replace(b"\0", b" ").decode(errors="replace")
        return pid if expected in cmdline else None
    except (OSError, ValueError):
        return None


def start_tunnel(work, get_secret):
    token = get_secret("CF_TUNNEL_TOKEN")
    if not token:
        print("⚠️ 服务已启动，但缺少 CF_TUNNEL_TOKEN。请在 Colab Secrets 添加并授权后重跑本格。", flush=True)
        return False
    pidfile = work / "studio-cloudflared.pid"
    if live_owned(pidfile, "cloudflared"):
        print("⏩ 本工作台隧道进程已在运行，连接状态见 studio-cloudflared.log", flush=True)
        return True
    binary = shutil.which("cloudflared")
    if not binary:
        local = Path("/content/cloudflared")
        if not local.is_file() or local.stat().st_size < 1024**2:
            partial = local.with_suffix(".part")
            machine = os.uname().machine
            arch = "arm64" if machine in {"aarch64", "arm64"} else "amd64"
            url = f"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{arch}"
            subprocess.run(["curl", "-fL", "--retry", "2", "--connect-timeout", "15", "--max-time", "120",
                            "-o", str(partial), url], check=True)
            partial.replace(local)
        local.chmod(0o755)
        binary = str(local)
    tunnel_env = os.environ.copy()
    tunnel_env["TUNNEL_TOKEN"] = token
    log = work / "logs" / "studio-cloudflared.log"
    with log.open("w", encoding="utf-8") as fp:
        # 使用环境变量传入令牌，避免在命令行与日志打印中暴露。
        proc = subprocess.Popen([binary, "tunnel", "--protocol", "http2", "run"],
                                stdout=fp, stderr=subprocess.STDOUT, env=tunnel_env, start_new_session=True)
    pidfile.write_text(str(proc.pid))
    start = time.monotonic()
    while time.monotonic()-start < 30:
        text = log.read_text(encoding="utf-8", errors="replace")
        if proc.poll() is not None:
            print("⚠️ 隧道退出，工作台仍在本地运行。日志：\n" + text[-8000:], flush=True)
            return False
        if "Registered tunnel connection" in text:
            print("✅ Cloudflare HTTP/2 隧道已注册连接", flush=True)
            return True
        time.sleep(.5)
    print(f"⚠️ 隧道连接尚未确认，请查看 {log}；本地工作台已就绪。", flush=True)
    return False


def main():
    if sys.platform != "linux":
        raise RuntimeError("本文件在 Colab A100 Linux 中运行；本地看界面请运行 preview.py")
    payload = json.loads(gzip.decompress(base64.b64decode(BUNDLE_GZ_B64)))
    namespace = {"__name__": "pika_installer", "__file__": "/content/pika-studio-installer.py"}
    exec(compile(payload.pop("installer_runtime.py"), "pika-studio-installer.py", "exec"), namespace)
    config = dict(namespace["CONFIG"], storage_root=STORAGE_ROOT, http_download=HTTP_DOWNLOAD)
    namespace["CONFIG"] = config
    current = health()
    if current.get("service") == "pika-h3-flex-studio":
        work = namespace["select_storage"](config)
        if current.get("version") == APP_VERSION:
            print("⏩ 已有同版本 Pika 工作台在线，无需密码，保留当前任务。", flush=True)
            start_tunnel(work, namespace["secret"])
            print(f"🌐 在电脑上打开工作台：{STUDIO_URL}\n后端隧道：{PUBLIC_URL}", flush=True)
            return
        # Only update our own idle server. Do not interrupt GPU work or queued jobs.
        for path in (work / "jobs").glob("*.json"):
            job = json.loads(path.read_text(encoding="utf-8"))
            if job.get("status") in {"running", "queued", "cancelling"}:
                raise RuntimeError("更新进度条前，请等当前队列完成或在网页取消任务，然后重跑这格；不会中断生成。")
        pid = live_owned(work / "studio-server.pid", str(work / "studio-app" / "server.py"))
        if not pid:
            raise RuntimeError("检测到旧版服务但无法确认归属，请先手动停止旧工作台服务器")
        import signal
        os.kill(pid, signal.SIGTERM)
        for _ in range(40):
            if not health():
                break
            time.sleep(.25)
        else:
            raise RuntimeError("旧版服务尚未停止，请稍后重跑")
        print("✅ 已停止本工作台的空闲旧版服务，正在更新。", flush=True)
    try:
        with socket.create_connection(("127.0.0.1", PORT), timeout=1):
            raise RuntimeError(f"{PORT} 已被其他程序占用，请先停止旧 ComfyUI，再运行工作台；本脚本不会杀其他进程。")
    except OSError:
        pass
    work, env, cli = namespace["prepare_environment"]()
    app_dir = work / "studio-app"
    for name, content in payload.items():
        target = app_dir / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(content, encoding="utf-8")
    env.pop("PIKA_STUDIO_PASSWORD", None)
    env.pop("CF_TUNNEL_TOKEN", None)
    log = work / "logs" / "studio-server.log"
    with log.open("a", encoding="utf-8") as fp:
        proc = subprocess.Popen([str(cli.parent / "python"), str(app_dir / "server.py"),
                                 "--root", str(work), "--port", str(PORT)],
                                stdout=fp, stderr=subprocess.STDOUT, env=env, start_new_session=True)
    (work / "studio-server.pid").write_text(str(proc.pid))
    for _ in range(120):
        if proc.poll() is not None:
            print(log.read_text(encoding="utf-8", errors="replace")[-10000:], flush=True)
            raise RuntimeError("工作台服务提前退出")
        if health().get("service") == "pika-h3-flex-studio":
            break
        time.sleep(.5)
    else:
        print(log.read_text(encoding="utf-8", errors="replace")[-10000:], flush=True)
        raise RuntimeError("工作台启动超时")
    print("✅ 工作台已启动，生成任务在网页提交，不会自动跑示例。", flush=True)
    print("🌐 工作台无需登录，打开网页即可使用。", flush=True)
    try:
        connected = start_tunnel(work, namespace["secret"])
    except Exception as exc:
        connected = False
        print(f"⚠️ 隧道准备失败：{exc}。本地工作台仍在运行。", flush=True)
    print(f"🌐 在电脑上打开工作台：{STUDIO_URL}\n后端隧道：{PUBLIC_URL}", flush=True)
    print(f"隧道 {'已连接' if connected else '尚待确认'}；Cloudflare 服务地址应为 http://127.0.0.1:{PORT}", flush=True)
    print(f"模型缓存复用：{work / 'hf'}\n服务日志：{log}\n生成视频：{work / 'outputs'}", flush=True)


if __name__ == "__main__":
    main()
